In [ ]:
# ============================================================
# STARTUP (Target-Agnostic)
# ============================================================
# Install required packages, import common libraries,
# mount Google Drive, configure project paths,
# load project config, and create common project directories.

!pip -q install "datasets<4.0.0" "huggingface_hub<0.25"

import io
import csv
import json
import time
import hashlib
import sys
import os
import importlib

from pathlib import Path
from typing import List, Dict, Optional, Tuple

from google.colab import drive
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

# Mount Google Drive
drive.mount('/content/drive')

# ----------------------------------------------------------------
# Standardized Drive / Project Paths
# ----------------------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'DIP_Project'
SRC_DIR = PROJECT_ROOT / 'src'

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Could not find project directory: {PROJECT_ROOT}')

if not SRC_DIR.exists():
    raise FileNotFoundError(f'Could not find src directory: {SRC_DIR}')

# Add project root and src to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# ----------------------------------------------------------------
# Load project config
# Expected file: /content/drive/MyDrive/DIP_Project/src/project_config.py
# ----------------------------------------------------------------
try:
    from project_config import *
    print('Project config loaded successfully.')
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'Could not import project_config.\n'
        'Expected file: /content/drive/MyDrive/DIP_Project/src/project_config.py'
    )

# ----------------------------------------------------------------
# Create common project directories
# ----------------------------------------------------------------
COMMON_DIRS = [
    'data/raw',
    'data/processed',
    'data/metadata',
    'outputs/images',
    'outputs/logs',
]

for d in COMMON_DIRS:
    os.makedirs(PROJECT_ROOT / d, exist_ok=True)

print('Startup complete.')
print('DRIVE_ROOT   :', DRIVE_ROOT)
print('PROJECT_ROOT :', PROJECT_ROOT)
print('SRC_DIR      :', SRC_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project config loaded successfully.
Startup complete.
DRIVE_ROOT   : /content/drive/MyDrive
PROJECT_ROOT : /content/drive/MyDrive/DIP_Project
SRC_DIR      : /content/drive/MyDrive/DIP_Project/src


In [ ]:
# ============================================================
# CELL 0: NOTEBOOK SUMMARY
# DATASET BUILDER DRIVER NOTEBOOK
# ============================================================

# Purpose:
#   - Build standardized image datasets from multiple sources
#   - Execute one dataset target at a time
#       (DiffusionDB, SDXL, ImageNet, COCO, Midjourney, OpenImages)
#   - Maintain a consistent, reusable pipeline across all sources

# Design Overview:
#   - This notebook implements a modular dataset builder pipeline
#   - Common workflow logic is centralized here
#   - Dataset-specific logic is encapsulated in target modules
#   - Shared project constants and paths are loaded from:
#         src/project_config.py

# Core Responsibilities (This Notebook):
#   - target selection and dynamic module loading
#   - directory and environment setup
#   - config-driven path and parameter usage
#   - CSV metadata creation and management
#   - image validation (size, format)
#   - hashing and duplicate detection
#   - deterministic filename generation
#   - image saving and metadata recording
#   - final dataset verification and inspection

# Target Module Responsibilities (src/targets/*.py):
#   - load_source_dataset(...)
#   - get_candidate_records(...)
#   - provide dataset-specific access and record extraction only

# Standard Candidate Record Format:
#   Each target module must return records with:
#     - source_id   : unique identifier from source dataset
#     - source_ref  : reference string (e.g., Hugging Face path or URL)
#     - image_obj   : PIL Image object

# Key Pipeline Features:
#   - Deterministic naming:
#       Filenames are based on accepted index (not filesystem state)
#       → ensures reproducibility and rebuild capability
#
#   - Idempotent execution:
#       Safe to rerun without duplicating work
#
#   - Resume support:
#       Existing images are reused when possible
#
#   - Metadata rebuild capability:
#       CSV can be regenerated from existing images without rebuilding dataset
#
#   - Data integrity guarantees:
#       Image save + CSV write are treated as a single atomic operation

# Valid TARGET_NAME values (specified in Cell 1):
#   - diffusiondb
#   - sdxl
#   - imagenet
#   - coco
#   - midjourney
#   - openimages

# Notes:
#   - The dataset source is selected in Cell 1 by setting TARGET_NAME
#       (e.g., 'diffusiondb', 'sdxl', 'imagenet', etc.)
#   - This notebook is otherwise fully target-agnostic and reusable
#   - Shared constants such as paths, image size, batch size, and target counts
#     are loaded from src/project_config.py
#   - Raw metadata CSV files are written to:
#         /content/drive/MyDrive/DIP_Project/data/metadata/
#     using the naming pattern:
#         [dataset_code]_raw_metadata.csv
#   - Large datasets are stored in Google Drive, not GitHub
#   - Google Drive is mounted in Colab under:
#         /content/drive/MyDrive/
#   - Target-specific source code is stored in src/targets/*.py
#   - An optional inspection cell may be used to display the imported
#     target-specific functions so their behavior is visible in Colab
#   - DiffusionDB may require:
#         trust_remote_code=True
#     in load_dataset(...) inside the target module to avoid prompts

# Performance Notes:
#   - MS COCO 2017 behaves differently from some other datasets:
#       • Images are fetched via HTTP (one request per image)
#       • This makes candidate extraction and dataset building slower
#
#   - DiffusionDB and ImageNet:
#       • Often provide image content more directly through the dataset interface
#       • Usually run faster per batch
#
#   - Recommended settings for slower sources:
#       • Use smaller batch sizes
#       • Expect longer execution times
#
#   - For debugging:
#       • Reduce TARGET_ACCEPTED (for example, 50)
#       • Use a smaller BATCH_SIZE (for example, 10)
#
#   - For full builds:
#       • Execution time is dominated by network download speed
#       • Performance may vary depending on source behavior and connection quality

# ============================================================
# Expected Google Drive / Colab Project Layout
# ============================================================
# /content/drive/MyDrive/
# └── DIP_Project/
#     ├── src/
#     │   ├── project_config.py
#     │   ├── targets/
#     │   │   ├── diffusiondb_target.py
#     │   │   ├── sdxl_target.py
#     │   │   ├── imagenet_target.py
#     │   │   ├── coco_target.py
#     │   │   ├── midjourney_target.py
#     │   │   └── openimages_target.py
#     │   └── utils/
#     │
#     ├── notebooks/
#     │   └── 01_Build_Dataset.ipynb
#     │
#     ├── data/
#     │   ├── raw/           # saved raw images by dataset
#     │   ├── preprocessed/  # optional later-stage image outputs
#     │   └── metadata/      # CSV + hash files
#     │
#     └── outputs/
#         ├── images/        # optional alternative output path
#         └── logs/          # logs, debug output


In [ ]:
# ============================================================
# Cell 1: Select Target Dataset
# ============================================================

TARGETS = {
    'diffusiondb': {
        'display_name': 'DiffusionDB',
        'module_name': 'targets.diffusiondb_target',
        'source_dataset': 'DiffusionDB',
        'filename_label': 'ai',
        'dataset_code': 'diff',
    },
    'sdxl': {
        'display_name': 'SDXL Generated (10K)',
        'module_name': 'targets.sdxl_target',
        'source_dataset': 'SDXL_Generated_10K',
        'filename_label': 'ai',
        'dataset_code': 'sdxl',
    },
    'imagenet': {
        'display_name': 'ImageNet (256)',
        'module_name': 'targets.imagenet_target',
        'source_dataset': 'ImageNet_1K_256',
        'filename_label': 'rl',
        'dataset_code': 'imgn',
    },
    'coco': {
        'display_name': 'MS COCO 2017',
        'module_name': 'targets.coco_target',
        'source_dataset': 'MS_COCO_2017',
        'filename_label': 'rl',
        'dataset_code': 'coco',
    },
    'midjourney': {
        'display_name': 'Midjourney',
        'module_name': 'targets.midjourney_target',
        'source_dataset': 'Midjourney',
        'filename_label': 'ai',
        'dataset_code': 'mj',
    },
    'openimages': {
        'display_name': 'Open Images',
        'module_name': 'targets.openimages_target',
        'source_dataset': 'OpenImages',
        'filename_label': 'rl',
        'dataset_code': 'open',
    },
}

# Change only this line to switch datasets.
TARGET_NAME = 'diffusiondb'   # diffusiondb, sdxl, imagenet, coco, midjourney, openimages

if TARGET_NAME not in TARGETS:
    valid_targets = ', '.join(TARGETS.keys())
    raise ValueError(
        f"Unknown TARGET_NAME: {TARGET_NAME}\n"
        f"Valid options are: {valid_targets}"
    )

TARGET_CONFIG = TARGETS[TARGET_NAME]

TARGET_DISPLAY_NAME = TARGET_CONFIG['display_name']
TARGET_MODULE_NAME  = TARGET_CONFIG['module_name']
SOURCE_DATASET      = TARGET_CONFIG['source_dataset']
FILENAME_LABEL      = TARGET_CONFIG['filename_label']
DATASET_CODE        = TARGET_CONFIG['dataset_code']

print(f'Target selected: {TARGET_NAME}')
print(f'Display name   : {TARGET_DISPLAY_NAME}')
print(f'Source dataset : {SOURCE_DATASET}')
print(f'Filename label : {FILENAME_LABEL}')
print(f'Dataset code   : {DATASET_CODE}')
print(f'Module         : {TARGET_MODULE_NAME}')


Target selected: diffusiondb
Display name   : DiffusionDB
Source dataset : DiffusionDB
Filename label : ai
Dataset code   : diff
Module         : targets.diffusiondb_target


In [ ]:
# ============================================================
# Cell 2: Load Target Module
# ============================================================

try:
    target_module = importlib.import_module(TARGET_MODULE_NAME)
    importlib.reload(target_module)
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        f"Could not import module: {TARGET_MODULE_NAME}\n"
        "Check that the file exists under src/targets/ and the path is correct."
    )

# Validate required interface
required_functions = ['load_source_dataset', 'get_candidate_records']

for func in required_functions:
    if not hasattr(target_module, func):
        raise AttributeError(
            f"Target module '{TARGET_MODULE_NAME}' is missing required function: {func}"
        )

# Bind functions
load_source_dataset = target_module.load_source_dataset
get_candidate_records = target_module.get_candidate_records

print(f'Imported target module for {TARGET_DISPLAY_NAME}: OK')
print('Module        :', TARGET_MODULE_NAME)
print('Module file   :', target_module.__file__)


Imported target module for DiffusionDB: OK
Module        : targets.diffusiondb_target
Module file   : /content/drive/MyDrive/DIP_Project/src/targets/diffusiondb_target.py


In [ ]:
# ============================================================
# OPTIONAL Cell 2A: Inspect Target Module Source Code
# ============================================================

import inspect

SHOW_SOURCE = False   # Set to True to display source code

if SHOW_SOURCE:
    print("Module file:", target_module.__file__)
    print("\n--- load_source_dataset() ---\n")
    print(inspect.getsource(load_source_dataset))

    print("\n--- get_candidate_records() ---\n")
    print(inspect.getsource(get_candidate_records))


In [ ]:
# ============================================================
# Cell 3: Common Configuration
# ============================================================

# Shared constants are imported from src/project_config.py
# in the Startup Cell. This cell derives dataset-specific
# paths from the selected TARGET_NAME / SOURCE_DATASET.

BASE_DIR = PROJECT_ROOT / 'data'
IMAGES_DIR = BASE_DIR / 'raw' / SOURCE_DATASET / 'images'
METADATA_DIR = BASE_DIR / 'metadata'
CSV_PATH = METADATA_DIR / f'{DATASET_CODE}_raw_metadata.csv'
SOURCE_HASH_PATH = METADATA_DIR / f'{SOURCE_DATASET.lower()}_source_hashes.json'
GLOBAL_HASH_PATH = METADATA_DIR / 'global_hashes.json'

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print('Resolved configuration:')
print('TARGET_ACCEPTED      :', TARGET_ACCEPTED)
print('MIN_WIDTH            :', MIN_WIDTH)
print('MIN_HEIGHT           :', MIN_HEIGHT)
print('BATCH_ID_START       :', BATCH_ID_START)
print('BATCH_SIZE           :', BATCH_SIZE)
print('RANDOM_SEED          :', RANDOM_SEED)
print('SLEEP_BETWEEN_BATCHES:', SLEEP_BETWEEN_BATCHES)
print('IMAGES_DIR           :', IMAGES_DIR)
print('METADATA_DIR         :', METADATA_DIR)
print('CSV_PATH             :', CSV_PATH)
print('SOURCE_HASH_PATH     :', SOURCE_HASH_PATH)
print('GLOBAL_HASH_PATH     :', GLOBAL_HASH_PATH)


Resolved configuration:
TARGET_ACCEPTED      : 3000
MIN_WIDTH            : 256
MIN_HEIGHT           : 256
BATCH_ID_START       : 1
BATCH_SIZE           : 200
RANDOM_SEED          : 42
SLEEP_BETWEEN_BATCHES: 1.0
IMAGES_DIR           : /content/drive/MyDrive/DIP_Project/data/raw/DiffusionDB/images
METADATA_DIR         : /content/drive/MyDrive/DIP_Project/data/metadata
CSV_PATH             : /content/drive/MyDrive/DIP_Project/data/metadata/diff_raw_metadata.csv
SOURCE_HASH_PATH     : /content/drive/MyDrive/DIP_Project/data/metadata/diffusiondb_source_hashes.json
GLOBAL_HASH_PATH     : /content/drive/MyDrive/DIP_Project/data/metadata/global_hashes.json


In [ ]:
# ============================================================
# Cell 4: CSV Setup
# ============================================================

CSV_COLUMNS = [
    'filename',
    'label',
    'dataset_code',
    'source_name',
    'source_id',
    'source_ref',
    'original_width',
    'original_height',
    'saved_path',
    'sha256',
    'batch_id',
]

def initialize_csv(csv_path: Path, columns: List[str]) -> None:
    if not csv_path.exists():
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()

initialize_csv(CSV_PATH, CSV_COLUMNS)
print('CSV initialized:', CSV_PATH)


CSV initialized: /content/drive/MyDrive/DIP_Project/data/metadata/diff_raw_metadata.csv


In [ ]:
# ============================================================
# Cell 5: Hash Utilities
# ============================================================

def load_hash_set(hash_path: Path) -> set:
    if hash_path.exists():
        with open(hash_path, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_hash_set(hash_path: Path, hash_set: set) -> None:
    with open(hash_path, 'w', encoding='utf-8') as f:
        json.dump(sorted(list(hash_set)), f, indent=2)

source_hashes = load_hash_set(SOURCE_HASH_PATH)
global_hashes = load_hash_set(GLOBAL_HASH_PATH)

print('Loaded source hashes:', len(source_hashes))
print('Loaded global hashes:', len(global_hashes))


Loaded source hashes: 0
Loaded global hashes: 0


In [ ]:
# ============================================================
# Cell 6: Filename and Counting Utilities
# ============================================================

def count_existing_images(images_dir: Path) -> int:
    return len(list(images_dir.glob('*.png')))

def next_index(images_dir: Path) -> int:
    return count_existing_images(images_dir) + 1

def build_filename(label: str, dataset_code: str, idx: int) -> str:
    return f'{label}_{dataset_code}_{idx:06d}.png'


In [ ]:
# ============================================================
# Cell 7: Image Utilities
# ============================================================

def normalize_image(img: Image.Image) -> Image.Image:
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return img

def image_meets_size_requirement(
    img: Image.Image,
    min_width: int = MIN_WIDTH,
    min_height: int = MIN_HEIGHT,
) -> bool:
    w, h = img.size
    return (w >= min_width) and (h >= min_height)

def compute_sha256_for_normalized_png(img: Image.Image) -> str:
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return hashlib.sha256(buffer.getvalue()).hexdigest()


In [ ]:
# ============================================================
# Cell 8: CSV Append Utility
# ============================================================

def append_csv_row(csv_path: Path, row: Dict) -> None:
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writerow(row)


In [ ]:
# ============================================================
# Cell 9: Load Source Dataset
# ============================================================

source_ds = load_source_dataset(seed=RANDOM_SEED)
print(f'{TARGET_DISPLAY_NAME} source dataset loaded.')
print('Type:', type(source_ds))
print('Current accepted images:', count_existing_images(IMAGES_DIR))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

diffusiondb.py: 0.00B [00:00, ?B/s]

part-000237.zip:   0%|          | 0.00/609M [00:00<?, ?B/s]

part-001479.zip:   0%|          | 0.00/709M [00:00<?, ?B/s]

...

part-000088.zip:   0%|          | 0.00/562M [00:00<?, ?B/s]

part-000151.zip:   0%|          | 0.00/617M [00:00<?, ?B/s]

metadata.parquet:   0%|          | 0.00/195M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DiffusionDB source dataset loaded.
Type: <class 'datasets.arrow_dataset.Dataset'>
Current accepted images: 0


In [ ]:
# ============================================================
# Cell 10: Preview Candidate Records
# ============================================================

candidate_records = get_candidate_records(
    source_ds,
    batch_size=BATCH_SIZE,
    batch_id=BATCH_ID_START,
)

print(f'Candidate records collected: {len(candidate_records)}')

if len(candidate_records) > 0:
    print('First candidate record:')
    print(candidate_records[0])


Candidate records collected: 200
First candidate record:
{'source_id': 'diffusiondb_0', 'source_ref': 'huggingface://poloclub/diffusiondb/2m_random_5k/0', 'image_obj': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1152x640 at 0x7BA7E8255820>}


In [ ]:
# ============================================================
# Cell 11: Process One Candidate Record
# ============================================================

def process_candidate(record: Dict, batch_id: int, idx: int) -> Tuple[bool, Optional[str]]:
    global source_hashes, global_hashes

    source_id = record.get('source_id', '')
    source_ref = record.get('source_ref', '')
    img = record.get('image_obj', None)

    if img is None:
        return False, 'missing_image'

    try:
        img = normalize_image(img)
    except Exception:
        return False, 'invalid_image'

    width, height = img.size

    if not image_meets_size_requirement(img):
        return False, 'too_small'

    try:
        sha256 = compute_sha256_for_normalized_png(img)
    except Exception:
        return False, 'hash_failed'

    if sha256 in source_hashes:
        return False, 'duplicate_within_source'

    if sha256 in global_hashes:
        return False, 'duplicate_global'

    filename = build_filename(FILENAME_LABEL, DATASET_CODE, idx)
    save_path = IMAGES_DIR / filename

    row = {
        'filename': filename,
        'label': FILENAME_LABEL,
        'dataset_code': DATASET_CODE,
        'source_name': SOURCE_DATASET,
        'source_id': source_id,
        'source_ref': source_ref,
        'original_width': width,
        'original_height': height,
        'saved_path': str(save_path),
        'sha256': sha256,
        'batch_id': batch_id,
    }

    try:
        image_already_exists = save_path.exists()

        if not image_already_exists:
            img.save(save_path, format='PNG')

        append_csv_row(CSV_PATH, row)

    except Exception:
        # If we created the image but CSV failed, clean it up
        try:
            if not image_already_exists and save_path.exists():
                save_path.unlink()
        except Exception:
            pass

        return False, 'save_or_csv_failed'

    source_hashes.add(sha256)
    global_hashes.add(sha256)

    return True, None


In [ ]:
# ============================================================
# Cell 12: Build Dataset for Selected Source
# ============================================================

import gc
import time
from tqdm import tqdm

def build_dataset_for_source(
    batch_size: int = BATCH_SIZE,
    sleep_between_batches: float = SLEEP_BETWEEN_BATCHES,
) -> None:
    batch_id = BATCH_ID_START
    accepted_total = len(source_hashes)

    print(f'Starting source: {TARGET_NAME}')
    print(f'Current accepted images: {accepted_total}')
    print(f'Target accepted images: {TARGET_ACCEPTED}')
    print('-' * 60)

    while accepted_total < TARGET_ACCEPTED:
        needed = TARGET_ACCEPTED - accepted_total
        print(f'\nBatch {batch_id} | accepted so far: {accepted_total} | still needed: {needed}')

        records = get_candidate_records(
            source_ds,
            batch_size=batch_size,
            batch_id=batch_id,
        )

        if not records:
            print('No more records returned by target module. Stopping.')
            break

        accepted_in_batch = 0
        rejected_counts = {}

        for record in tqdm(records, desc=f'Batch {batch_id}'):
            next_idx = accepted_total + 1
            img = record.get("image_obj", None)

            try:
                accepted, reason = process_candidate(record, batch_id, next_idx)

                if accepted:
                    accepted_in_batch += 1
                    accepted_total += 1

                    if accepted_total >= TARGET_ACCEPTED:
                        break
                else:
                    rejected_counts[reason] = rejected_counts.get(reason, 0) + 1

            finally:
                if img is not None:
                    try:
                        img.close()
                    except Exception:
                        pass
                record["image_obj"] = None

        save_hash_set(SOURCE_HASH_PATH, source_hashes)
        save_hash_set(GLOBAL_HASH_PATH, global_hashes)

        print(f'Accepted this batch: {accepted_in_batch}')
        if rejected_counts:
            print('Rejected counts:', rejected_counts)

        del records
        gc.collect()

        batch_id += 1
        if accepted_total < TARGET_ACCEPTED:
            time.sleep(sleep_between_batches)

    print('\nBuild complete.')
    print(f'Final accepted images: {accepted_total}')


In [ ]:
# ============================================================
# Cell 13: Verify Saved Output
# ============================================================

def csv_row_count(csv_path: Path) -> int:
    if not csv_path.exists():
        return 0
    with open(csv_path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)

def verify_source_output() -> None:
    image_count = count_existing_images(IMAGES_DIR)
    row_count = csv_row_count(CSV_PATH)

    print(f'Source: {SOURCE_DATASET}')
    print(f'Image count: {image_count}')
    print(f'CSV row count: {row_count}')

    if image_count == row_count:
        print('PASS: image count matches CSV row count')
    else:
        print('FAIL: image count does not match CSV row count')


In [ ]:
# ============================================================
# Cell 14: Run Builder
# ============================================================

build_dataset_for_source(batch_size=BATCH_SIZE)


Starting source: diffusiondb
Current accepted images: 0
Target accepted images: 3000
------------------------------------------------------------

Batch 1 | accepted so far: 0 | still needed: 3000


Batch 1: 100%|██████████| 200/200 [00:58<00:00,  3.43it/s]


Accepted this batch: 200

Batch 2 | accepted so far: 200 | still needed: 2800


Batch 2: 100%|██████████| 200/200 [00:56<00:00,  3.55it/s]


Accepted this batch: 200

Batch 3 | accepted so far: 400 | still needed: 2600


...

Accepted this batch: 199
Rejected counts: {'duplicate_within_source': 1}

Batch 15 | accepted so far: 2793 | still needed: 207


Batch 15: 100%|██████████| 200/200 [01:00<00:00,  3.32it/s]


Accepted this batch: 198
Rejected counts: {'too_small': 1, 'duplicate_within_source': 1}

Batch 16 | accepted so far: 2991 | still needed: 9


Batch 16:   4%|▍         | 8/200 [00:02<01:01,  3.12it/s]


Accepted this batch: 9

Build complete.
Final accepted images: 3000


In [ ]:
# ============================================================
# Cell 15: Final Verification
# ============================================================

verify_source_output()


Source: DiffusionDB
Image count: 3000
CSV row count: 3000
PASS: image count matches CSV row count


In [ ]:
# ============================================================
# Rebuild global_hashes.json from all source hash files
# ============================================================

import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DIP_Project")
METADATA_DIR = PROJECT_ROOT / "data/metadata"

SOURCE_HASH_FILES = [
    METADATA_DIR / "diff_source_hashes.json",
    METADATA_DIR / "sdxl_source_hashes.json",
    METADATA_DIR / "imgn_source_hashes.json",
    METADATA_DIR / "coco_source_hashes.json",
    METADATA_DIR / "mj_source_hashes.json",
    METADATA_DIR / "open_source_hashes.json",
]

all_hashes = set()
total_entries = 0

for path in SOURCE_HASH_FILES:
    if not path.exists():
        raise FileNotFoundError(f"Missing source hash file: {path}")

    with open(path, "r") as f:
        data = json.load(f)   # {filename: hash}

    count = len(data)
    total_entries += count
    all_hashes.update(data.values())

    print(f"{path.name:<35} {count}")

GLOBAL_HASH_FILE = METADATA_DIR / "global_hashes.json"

with open(GLOBAL_HASH_FILE, "w") as f:
    json.dump(sorted(all_hashes), f, indent=2)

print("\n----------------------------------------")
print(f"Total input hashes : {total_entries}")
print(f"Unique global hashes: {len(all_hashes)}")
print(f"Saved: {GLOBAL_HASH_FILE}")

if total_entries == 18000 and len(all_hashes) == 18000:
    print("PASS: all 18000 images are globally unique")
else:
    print("NOTE: global uniqueness is less than 18000; investigate if unexpected")


diff_source_hashes.json             3000
sdxl_source_hashes.json             3000
imgn_source_hashes.json             3000
coco_source_hashes.json             3000
mj_source_hashes.json               3000
open_source_hashes.json             3000

----------------------------------------
Total input hashes : 18000
Unique global hashes: 18000
Saved: /content/drive/MyDrive/DIP_Project/data/metadata/global_hashes.json
PASS: all 18000 images are globally unique
